In [2]:
#Import dependencies
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pmdarima as pm
from pmdarima.arima import ndiffs, nsdiffs
from pmdarima.model_selection import cross_val_score, RollingForecastCV
from sklearn.preprocessing import StandardScaler
from statsmodels.graphics.tsaplots import plot_ccf



In [ ]:
#Fetch data
LAT, LON = 56.1629, 10.2039  # Aarhus
START, END = "2022-01-01", "2026-04-18"

# ── 1. Historical air quality (NO2, O3) ──────────────────────────────────────
print("Fetching air quality...")
aq = requests.get(
    "https://air-quality-api.open-meteo.com/v1/air-quality",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "nitrogen_dioxide,ozone,pm10,pm2_5",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

aq_df = pd.DataFrame({
    "Recorded": pd.to_datetime(aq["hourly"]["time"]),
    "NO2":      aq["hourly"]["nitrogen_dioxide"],
    "O3":       aq["hourly"]["ozone"],
    "PM10":     aq["hourly"]["pm10"],
    "PM2.5":    aq["hourly"]["pm2_5"],
})

# ── 2. Historical weather (wind, rain, temp, radiation) ──────────────────────
print("Fetching weather...")
wx = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": LAT,
        "longitude": LON,
        "hourly": "wind_speed_10m,wind_direction_10m,temperature_2m,shortwave_radiation,precipitation,relative_humidity_2m",
        "wind_speed_unit": "ms",
        "start_date": START,
        "end_date": END,
        "timezone": "Europe/Copenhagen"
    }
).json()

wx_df = pd.DataFrame({
    "Recorded":           pd.to_datetime(wx["hourly"]["time"]),
    "wind_speed":         wx["hourly"]["wind_speed_10m"],
    "wind_direction":     wx["hourly"]["wind_direction_10m"],
    "temperature":        wx["hourly"]["temperature_2m"],
    "solar_radiation":    wx["hourly"]["shortwave_radiation"],
    "precipitation":      wx["hourly"]["precipitation"],
    "humidity":           wx["hourly"]["relative_humidity_2m"],

})

# ── 3. Merge and save ─────────────────────────────────────────────────────────
print("Merging...")
df = aq_df.merge(wx_df, on="Recorded", how="inner")
df = df.sort_values("Recorded").reset_index(drop=True)

print(f"\nFinal shape: {df.shape}")
print(df.head())
print(f"\nDate range: {df.Recorded.min()} → {df.Recorded.max()}")
print(f"Missing values:\n{df.isnull().sum()}")

#df.to_csv("aarhus_air_quality.csv", index=False)
#print("\nSaved to aarhus_air_quality.csv")

Fetching air quality...
Fetching weather...
Merging...

Final shape: (7320, 11)
             Recorded  NO2    O3  PM10  PM2.5  wind_speed  wind_direction  \
0 2025-06-18 00:00:00  2.7  69.0  13.9   10.4        5.58             264   
1 2025-06-18 01:00:00  2.5  68.0  13.2    9.1        5.17             266   
2 2025-06-18 02:00:00  2.5  68.0  12.5    9.1        5.35             268   
3 2025-06-18 03:00:00  2.2  71.0  11.9    9.3        5.20             270   
4 2025-06-18 04:00:00  2.2  69.0  11.7    8.5        5.17             274   

   temperature  solar_radiation  precipitation  humidity  
0         14.9              0.0            0.0        85  
1         14.9              0.0            0.0        85  
2         14.9              0.0            0.0        86  
3         14.7              0.0            0.0        88  
4         14.6              0.0            0.0        88  

Date range: 2025-06-18 00:00:00 → 2026-04-18 23:00:00
Missing values:
Recorded           0
NO2        

In [28]:
df.head()

,Recorded,NO2,O3,PM10,PM2.5,wind_speed,wind_direction,temperature,solar_radiation,precipitation,humidity
0,2022-01-01 00:00:00,6.2,53.0,6.2,5.0,5.00,269,6.7,0.0,0.0,98
1,2022-01-01 01:00:00,4.6,53.0,6.4,4.7,5.00,272,7.2,0.0,0.0,98
2,2022-01-01 02:00:00,5.0,57.0,6.0,4.5,4.90,269,7.1,0.0,0.0,97
3,2022-01-01 03:00:00,5.0,54.0,5.4,4.6,4.72,265,7.1,0.0,0.0,97
4,2022-01-01 04:00:00,5.7,52.0,5.3,4.9,4.53,264,7.1,0.0,0.0,96


In [4]:
# Setup
df["Recorded"] = pd.to_datetime(df["Recorded"])
df = df.set_index('Recorded')


target_col = 'NO2'
m = 24  # seasonal period (hourly data)

y = df[target_col].values
X = df.drop(columns=[target_col, "O3", "PM2.5", "PM10"]).values

# train/test split — hold out last 4 days (96 hours)
holdout = 96
y_train, y_test = y[:-holdout], y[-holdout:]
X_train, X_test = X[:-holdout], X[-holdout:]

In [36]:
X_train

array([[  5.  , 269.  ,   6.7 ,   0.  ,   0.  ,  98.  ],
       [  5.  , 272.  ,   7.2 ,   0.  ,   0.  ,  98.  ],
       [  4.9 , 269.  ,   7.1 ,   0.  ,   0.  ,  97.  ],
       ...,
       [  3.39, 261.  ,   7.3 ,   2.  ,   0.  ,  78.  ],
       [  2.77, 253.  ,   6.4 ,   0.  ,   0.  ,  84.  ],
       [  2.78, 247.  ,   5.2 ,   0.  ,   0.  ,  88.  ]], shape=(37560, 6))

In [8]:
#Scale exogenous features:

# Create scaler
X_scaler = StandardScaler()

# Fit on training data only
X_train_scaled = X_scaler.fit_transform(X_train)

# Use same scaling parameters on test data
X_test_scaled = X_scaler.transform(X_test)

In [9]:
X_train_scaled

array([[ 0.44843028,  0.80853235,  0.84961853, -0.59365376, -0.21668922,
         0.26930154],
       [ 0.25321409,  0.83290436,  0.84961853, -0.59365376, -0.21668922,
         0.26930154],
       [ 0.33891876,  0.85727637,  0.84961853, -0.59365376, -0.21668922,
         0.34616275],
       ...,
       [-0.59430987,  0.77197433, -0.20211334, -0.58248168, -0.21668922,
        -0.26872699],
       [-0.88951484,  0.67448629, -0.32666054, -0.59365376, -0.21668922,
         0.19244032],
       [-0.88475347,  0.60137026, -0.49272346, -0.59365376, -0.21668922,
         0.49988519]], shape=(7224, 6))

In [6]:
# Determine d and D
d = ndiffs(y_train, test='kpss')
D = nsdiffs(y_train, m=m, test='ch')
print(f"d={d}, D={D}")

d=1, D=1


In [ ]:

def seasonal_diff(arr, m):
    """Subtract value m steps ago from each observation"""
    if arr.ndim == 1:
        return arr[m:] - arr[:-m]
    else:
        return arr[m:] - arr[:-m]  # works for 2D arrays too (row-wise)

# apply non-seasonal diff first, then seasonal diff
y_train_diff = seasonal_diff(np.diff(y_train, n=d), m=24)
X_train_diff = seasonal_diff(np.diff(X_train_scaled, axis=0), m=24)

In [ ]:
feature_cols = ["wind_speed", "wind_direction", "temperature", "solar_radiation", "precipitation", "humidity"]
target_col = "NO2"

fig, axes = plt.subplots(len(feature_cols), 1, figsize=(12, 3 * len(feature_cols)))

for i, col in enumerate(feature_cols):
    plot_ccf(y_train_diff, X_train_diff[:, i], lags=48, ax=axes[i], negative_lags=False, alpha = 0.05)
    axes[i].set_title(f'CCF (differenced): {target_col} vs {col}')

plt.tight_layout()
plt.show()

In [ ]:
# Fit AutoARIMA 
model = pm.AutoARIMA(
    d=d, D=D,
    m=m,
    start_p=1, max_p=2,
    start_q=0, max_q=2,
    start_P=0, max_P=1,
    start_Q=0, max_Q=1,
    max_order=4,
    seasonal=True,
    stepwise=True,
    information_criterion='aic',
    suppress_warnings=True,
    error_action='ignore',
    trace=True  # prints models being tried
)
model.fit(y_train, X_train_scaled)
print(model.summary())

In [ ]:
# Residual diagnostics 
residuals = model.resid()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# residuals over time
axes[0, 0].plot(residuals)
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].set_title('Residuals over time')

# histogram
axes[0, 1].hist(residuals, bins=40)
axes[0, 1].set_title('Residual distribution')

# ACF of residuals
pm.utils.plot_acf(residuals, ax=axes[1, 0], lags=48)
axes[1, 0].set_title('ACF of residuals')

# PACF of residuals
pm.utils.plot_pacf(residuals, ax=axes[1, 1], lags=48)
axes[1, 1].set_title('PACF of residuals')

plt.tight_layout()
plt.show()

# Ljung-Box test — p-value should be > 0.05 (no autocorrelation left)
from statsmodels.stats.diagnostic import acorr_ljungbox
lb_test = acorr_ljungbox(residuals, lags=[24], return_df=True)
print(lb_test)


In [ ]:
# Cross-validation
cv = RollingForecastCV(h=24, step=12)

scores = cross_val_score(
    model, y_train,
    X=X_train,
    cv=cv,
    scoring='mean_absolute_error'
)
print(f"CV MAE scores: {scores}")
print(f"Mean CV MAE:   {np.mean(scores):.4f}")

In [ ]:
# Final evaluation on hold-out
# refit on full training data
model.fit(y_train, X=X_train)
forecast = model.predict(n_periods=holdout, X=X_test)

mae = np.mean(np.abs(forecast - y_test))
print(f"Hold-out MAE: {mae:.4f}")

# plot forecast vs actuals
plt.figure(figsize=(12, 4))
plt.plot(y_test, label='Actual')
plt.plot(forecast, label='Forecast', linestyle='--')
plt.legend()
plt.title('Hold-out forecast vs actuals')
plt.show()